In [87]:
# ==========================================
# IMPORTS
# ==========================================

from pathlib import Path
from dotenv import load_dotenv

from sentence_transformers import SentenceTransformer
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_groq import ChatGroq
from langchain_openai import ChatOpenAI


import pandas as pd
import numpy as np
import faiss
import json
import re
import os

# ==========================================
# LOAD ENVIRONMENT VARIABLES
# ==========================================

load_dotenv(Path("../.env"), override=True)

GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")
GROQ_API_KEY = os.getenv("GROQ_API_KEY")
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")

print("Google key loaded:", GOOGLE_API_KEY is not None)
print("Groq key loaded:", GROQ_API_KEY is not None)

# ==========================================
# PROJECT PATHS
# ==========================================

BASE_DIR = Path(".")
DATA_DIR = BASE_DIR / "data"

TRANSCRIPTS_DIR = DATA_DIR / "Transcripts"
QA_DIR = DATA_DIR / "QA"

# ==========================================
# LOAD FILE LISTS
# ==========================================

transcript_files = sorted(
    TRANSCRIPTS_DIR.glob("*.txt")
)

qa_files = sorted(
    QA_DIR.glob("*.json")
)

print("Transcripts folder exists:", TRANSCRIPTS_DIR.exists())
print("QA folder exists:", QA_DIR.exists())

print("Number of transcript files:", len(transcript_files))
print("Number of QA files:", len(qa_files))

Google key loaded: True
Groq key loaded: True
Transcripts folder exists: True
QA folder exists: True
Number of transcript files: 13
Number of QA files: 7


In [14]:
# ==========================================
# LOAD TRANSCRIPT FILES
# ==========================================

def load_transcripts(transcript_files):
    """
    Loads transcript text files and converts them
    into a structured pandas DataFrame.

    Expected line format:
    timestamp: text
    """

    rows = []

    for file_path in transcript_files:

        episode_name = file_path.stem

        with open(file_path, "r", encoding="utf-8") as f:

            for line in f:

                line = line.strip()

                # Skip empty lines
                if not line:
                    continue

                # Match timestamp + text
                match = re.match(
                    r"^([\d.]+):\s*(.*)$",
                    line
                )

                if match:

                    timestamp = float(match.group(1))

                    text = match.group(2).strip()

                    # Skip empty transcript text
                    if not text:
                        continue

                    rows.append({
                        "episode": episode_name,
                        "timestamp": timestamp,
                        "text": text
                    })

    return pd.DataFrame(rows)

# ==========================================
# LOAD TRANSCRIPTS DATAFRAME
# ==========================================

transcripts_df = load_transcripts(
    transcript_files
)

print("Total transcript rows:", len(transcripts_df))

display(transcripts_df.head())

# ==========================================
# TRANSCRIPT STATISTICS
# ==========================================

display(
    transcripts_df.groupby("episode")
    .size()
    .reset_index(name="num_lines")
)

Total transcript rows: 11109


,episode,timestamp,text
0,أعظم طائرة حربية الدحيح,0.000,سيادة الكولونيل، صبرك في محله،
1,أعظم طائرة حربية الدحيح,3.076,مبروك علينا،
2,أعظم طائرة حربية الدحيح,4.238,"عملنا أفجر طيارة في تاريخ ""أمريكا""."
3,أعظم طائرة حربية الدحيح,6.184,أنا متحمس جدًا من امبارح،
4,أعظم طائرة حربية الدحيح,8.308,ها، ورّيني!


,episode,num_lines
0,أعظم طائرة حربية الدحيح,761
1,الأخطبوط الدحيح,784
2,الساموراي الدحيح,662
3,تاج محل الدحيح,547
4,جون كينيدي الدحيح,1125
5,فيزياء و فلسفة الحركة الدحيح,955
6,كيف تحولت روسيا إلى إمبراطورية؟ الدحيح,1166
7,كيف تسيطر على عقول البشر؟ الدحيح,908
8,كيف تنقل جبل وزنه 30 طن قبل أن يغرق؟ الدحيح,757
9,مصير الأرض و الشمس و كل شيء الدحيح,656


In [15]:
# ==========================================
# LIMIT DATASET TO SELECTED EPISODES
# ==========================================

SELECTED_EPISODES = [
    "أعظم طائرة حربية  الدحيح",
    "الساموراي  الدحيح",
    "هل Citizen Kane أفضل فيلم في التاريخ؟  الدحيح",
    "الأخطبوط  الدحيح"
]

transcripts_df = transcripts_df[
    transcripts_df["episode"].isin(SELECTED_EPISODES)
].reset_index(drop=True)

print("Selected episodes:", transcripts_df["episode"].nunique())
print("Rows after filtering:", len(transcripts_df))

display(
    transcripts_df.groupby("episode")
    .size()
    .reset_index(name="num_lines")
)

Selected episodes: 4
Rows after filtering: 2924


,episode,num_lines
0,أعظم طائرة حربية الدحيح,761
1,الأخطبوط الدحيح,784
2,الساموراي الدحيح,662
3,هل Citizen Kane أفضل فيلم في التاريخ؟ الدحيح,717


In [16]:
# ==========================================
# MS3-SAFE TEXT NORMALIZATION
# ==========================================

"""
Normalization constraints for MS3:
- No stemming
- No lemmatization
- No punctuation removal
- No English token removal
- Preserve dialectal Arabic and Arabic-English code-switching
"""

AR_PUNCT_MAP = {
    ",": "،",
    ";": "؛",
    "?": "؟",
    "\"": "«",
    "“": "«",
    "”": "»"
}

PUNCT_RE = re.compile(
    "|".join(re.escape(k) for k in AR_PUNCT_MAP.keys())
)

def remove_noise_tags(text: str) -> str:
    """
    Removes non-speech tags such as [موسيقى].
    This does not remove actual spoken content.
    """
    return re.sub(r"\[.*?\]", "", text).strip()


def normalize_ms3_text(text: str) -> str:
    """
    Applies light normalization suitable for MS3 RAG.

    The goal is to clean formatting while preserving
    the natural transcript language.
    """

    if not isinstance(text, str):
        return ""

    # Remove non-speech tags only
    text = remove_noise_tags(text)

    # Standardize selected punctuation without removing punctuation
    text = PUNCT_RE.sub(
        lambda m: AR_PUNCT_MAP[m.group(0)],
        text
    )

    # Add spaces between Arabic and English/numeric tokens
    text = re.sub(
        r"([\u0600-\u06FF])([A-Za-z\d])",
        r"\1 \2",
        text
    )

    text = re.sub(
        r"([A-Za-z\d])([\u0600-\u06FF])",
        r"\1 \2",
        text
    )

    # Normalize whitespace only
    text = re.sub(r"\s+", " ", text).strip()

    return text


# Apply normalization
transcripts_df["normalized_text"] = transcripts_df["text"].apply(
    normalize_ms3_text
)

print(
    "normalized_text column created:",
    "normalized_text" in transcripts_df.columns
)

display(
    transcripts_df[
        ["episode", "timestamp", "text", "normalized_text"]
    ].head()
)

print("Original sample:")
print(transcripts_df["text"].iloc[0])

print("\nNormalized sample:")
print(transcripts_df["normalized_text"].iloc[0])

normalized_text column created: True


,episode,timestamp,text,normalized_text
0,أعظم طائرة حربية الدحيح,0.000,سيادة الكولونيل، صبرك في محله،,سيادة الكولونيل، صبرك في محله،
1,أعظم طائرة حربية الدحيح,3.076,مبروك علينا،,مبروك علينا،
2,أعظم طائرة حربية الدحيح,4.238,"عملنا أفجر طيارة في تاريخ ""أمريكا"".",عملنا أفجر طيارة في تاريخ «أمريكا«.
3,أعظم طائرة حربية الدحيح,6.184,أنا متحمس جدًا من امبارح،,أنا متحمس جدًا من امبارح،
4,أعظم طائرة حربية الدحيح,8.308,ها، ورّيني!,ها، ورّيني!


Original sample:
سيادة الكولونيل، صبرك في محله،

Normalized sample:
سيادة الكولونيل، صبرك في محله،


In [17]:
# ==========================================
# CHUNKING STRATEGY
# ==========================================

CHUNK_SIZE = 12
CHUNK_OVERLAP = 3
MIN_WORDS = 20

def create_chunks(transcripts_df, chunk_size=12, chunk_overlap=3, min_words=20):
    """
    Creates overlapping chunks from transcript lines.

    Each chunk remains traceable to:
    - source episode
    - start timestamp
    - end timestamp
    """

    chunks = []

    for episode in transcripts_df["episode"].unique():

        episode_df = transcripts_df[
            transcripts_df["episode"] == episode
        ].reset_index(drop=True)

        texts = episode_df["normalized_text"].tolist()
        timestamps = episode_df["timestamp"].tolist()

        start = 0

        while start < len(texts):

            end = min(start + chunk_size, len(texts))

            chunk_text = " ".join(texts[start:end])

            num_words = len(chunk_text.split())

            if num_words >= min_words:
                chunks.append({
                    "episode": episode,
                    "start_timestamp": timestamps[start],
                    "end_timestamp": timestamps[end - 1],
                    "chunk_text": chunk_text,
                    "num_words": num_words,
                    "num_chars": len(chunk_text)
                })

            start += chunk_size - chunk_overlap

    return pd.DataFrame(chunks)


chunks_df = create_chunks(
    transcripts_df,
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    min_words=MIN_WORDS
)

print("Total chunks:", len(chunks_df))
print("Average words per chunk:", round(chunks_df["num_words"].mean(), 2))
print("Minimum words:", chunks_df["num_words"].min())
print("Maximum words:", chunks_df["num_words"].max())

display(chunks_df.head())

Total chunks: 326
Average words per chunk: 60.96
Minimum words: 29
Maximum words: 79


,episode,start_timestamp,end_timestamp,chunk_text,num_words,num_chars
0,أعظم طائرة حربية الدحيح,0.000,25.494,سيادة الكولونيل، صبرك في محله، مبروك علينا، عم...,58,320
1,أعظم طائرة حربية الدحيح,22.165,47.616,يعني إيه سنين ضوئية؟! مش مهم، مش مهم، احكيلي ع...,62,339
2,أعظم طائرة حربية الدحيح,44.340,63.501,لو افترضنا إن هناك شخص، ومثلًا مثلًا مثلًا، يع...,57,319
3,أعظم طائرة حربية الدحيح,60.783,80.211,وصعب أي فرد يتتبعها على الـ... سؤال من واحد صا...,51,266
4,أعظم طائرة حربية الدحيح,77.405,97.433,ثانية واحدة! إيه 7 لغات دي؟! اوعى يكون فيها لغ...,55,287


In [18]:
sample_chunk = chunks_df.iloc[3]

print("Episode:")
print(sample_chunk["episode"])

print("\nTimestamps:")
print(sample_chunk["start_timestamp"], "->", sample_chunk["end_timestamp"])

print("\nChunk:")
print(sample_chunk["chunk_text"])

Episode:
أعظم طائرة حربية  الدحيح

Timestamps:
60.783 -> 80.211

Chunk:
وصعب أي فرد يتتبعها على الـ... سؤال من واحد صاحبي، اللي هو بيفكر يشتري الـ Model دا يعني. صاحبك؟! اتفضل. هل النظام بتاعها بالإنجليزي بس؟ باللهجة العبرية؟ ينفع، هي فيها 7 لغات. إيه دا؟! ثانية واحدة! إيه 7 لغات دي؟! اوعى يكون فيها لغة روسي! لأ، لأ، ما تخافش، ما فيهاش.


In [24]:
# ==========================================
# EMBEDDING MODEL
# ==========================================

EMBEDDING_MODEL_NAME = (
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

embedding_model = SentenceTransformer(
    EMBEDDING_MODEL_NAME
)

# ==========================================
# GENERATE EMBEDDINGS
# ==========================================

texts = chunks_df["chunk_text"].tolist()

embeddings = embedding_model.encode(
    texts,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print("Embeddings shape:", embeddings.shape)

# ==========================================
# BUILD FAISS VECTOR STORE
# ==========================================

embedding_dim = embeddings.shape[1]

index = faiss.IndexFlatIP(embedding_dim)

index.add(embeddings)

print("Vectors stored in FAISS:", index.ntotal)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/11 [00:00<?, ?it/s]

Embeddings shape: (326, 384)
Vectors stored in FAISS: 326


In [ ]:
# ==========================================
# KEYWORD-AWARE RETRIEVAL FUNCTION
# ==========================================

def extract_query_terms(query):
    """
    Extracts simple Arabic/English terms from the query.
    Used only for reranking retrieved semantic candidates.
    """

    query = query.lower()

    tokens = re.findall(
        r"[\u0600-\u06FFA-Za-z0-9]+",
        query
    )

    tokens = [
        token for token in tokens
        if len(token) > 2
    ]

    return set(tokens)


def keyword_score(query_terms, text):
    """
    Computes how many query terms appear in the chunk text.
    """

    text = text.lower()

    if not query_terms:
        return 0

    matches = sum(
        1 for term in query_terms
        if term in text
    )

    return matches / len(query_terms)


def retrieve_chunks(
    query,
    top_k=5,
    fetch_k=30,
    min_score=0.25,
    episode_filter=None
):
    """
    Hybrid retrieval:
    1. FAISS retrieves semantic candidates.
    2. Keyword overlap reranks the candidates.
    3. Final top_k chunks are returned.
    """

    query_embedding = embedding_model.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    scores, indices = index.search(
        query_embedding,
        fetch_k
    )

    query_terms = extract_query_terms(query)

    results = []

    for semantic_score, idx in zip(scores[0], indices[0]):

        row = chunks_df.iloc[idx]

        if float(semantic_score) < min_score:
            continue

        if episode_filter is not None and row["episode"] != episode_filter:
            continue

        k_score = keyword_score(
            query_terms,
            row["chunk_text"]
        )

        final_score = (
            0.70 * float(semantic_score)
        ) + (
            0.30 * k_score
        )

        results.append({
            "semantic_score": float(semantic_score),
            "keyword_score": k_score,
            "final_score": final_score,
            "episode": row["episode"],
            "start_timestamp": row["start_timestamp"],
            "end_timestamp": row["end_timestamp"],
            "chunk_text": row["chunk_text"]
        })

    results = sorted(
        results,
        key=lambda x: x["final_score"],
        reverse=True
    )

    return results[:top_k]


# ==========================================
# CONTEXT CONSTRUCTION
# ==========================================

def build_context(retrieved_chunks):
    """
    Builds a structured context string from retrieved chunks.
    Each chunk keeps its episode and timestamp for traceability.
    """

    context_parts = []

    for i, chunk in enumerate(retrieved_chunks, start=1):

        source = (
            f"[Source {i}] "
            f"Episode: {chunk['episode']} | "
            f"Time: {chunk['start_timestamp']} - "
            f"{chunk['end_timestamp']}"
        )

        text = chunk["chunk_text"]

        context_parts.append(
            source + "\n" + text
        )

    return "\n\n".join(context_parts)

In [26]:
# ==========================================
# OUT-OF-DOMAIN DETECTION
# ==========================================

OOD_THRESHOLD = 0.30

def is_out_of_domain(retrieved_chunks, threshold=OOD_THRESHOLD):
    """
    Detects whether the query is outside the transcript knowledge base.

    If no retrieved chunks are found, or the best retrieval score is too low,
    the query is considered out-of-domain.
    """

    if not retrieved_chunks:
        return True

    best_score = retrieved_chunks[0].get("final_score", 0)

    return best_score < threshold

In [29]:
query = "كيف كانت فلسفة الساموراي تجاه الموت؟"

results = retrieve_chunks(
    query,
    top_k=5,
    fetch_k=50,
    min_score=0.25,
    
)

for i, result in enumerate(results, start=1):
    print(f"\nResult {i}")
    print("Semantic score:", round(result["semantic_score"], 4))
    print("Keyword score:", round(result["keyword_score"], 4))
    print("Final score:", round(result["final_score"], 4))
    print("Episode:", result["episode"])
    print("Time:", result["start_timestamp"], "->", result["end_timestamp"])
    print(result["chunk_text"][:700])


Result 1
Semantic score: 0.738
Keyword score: 0.5
Final score: 0.6666
Episode: الساموراي  الدحيح
Time: 990.401 -> 1019.505
ودي كانت فلسفة حياة الساموراي بالظبط، «عيش اللحظة، اخدم سيدك بكل قوّتك، وموت وانت في قمة مجدك وشبابك.« مش محتاج، يا عزيزي، أقولّك الساموراي لو خسر في حرب، الحل الوحيد عشان يستعيد كرامته طقس Seppuku ، اللي بدأنا بيه الحلقة، الساموراي بيلبس كيمونو أبيض، ويقعد في هدوء وسَكينة، ويقوم كاتب قصيدة وداع أخيرة، وبعد كدا، يقوم مطلّع ويقوم غارزه في جنبه الشِمال،

Result 2
Semantic score: 0.5643
Keyword score: 0.5
Final score: 0.545
Episode: الساموراي  الدحيح
Time: 964.493 -> 995.231
وفي نفس الوقت، مُسالم ومُرهَف الحس. كانوا مؤمنين، يا عزيزي، العقل القوي هو اللي بيحرك السيف القوي، ودا خلّى طبقة الساموراي دي كمان كانت الطبقة المثقفة، اللي قادت «اليابان« وعشان كدا، الرمز المفضّل للساموراي زهرة رقيقة وجميلة، بس عُمرها قصير، بتفتح أيام قليلة، ودي كانت فلسفة حياة الساموراي بالظبط، «عيش اللحظة، اخدم سيدك بكل قوّتك، وموت وانت في قمة مجدك وشبابك.«

Result 3
Semantic score: 0.5105
Key

In [30]:
query = "ما هي تقنية Deep Focus؟"

retrieved = retrieve_chunks(query, top_k=3)

context = build_context(retrieved)

print(context[:3000])

[Source 1] Episode: هل Citizen Kane أفضل فيلم في التاريخ؟  الدحيح | Time: 742.927 - 768.418
عشان كدا، الـ Deep Focus لم تعُد دي بقت فلسفة كاملة في الإخراج، العمق البصري بقى مرادف للعمق النفسي، الصورة بقت مراية للعقل، مش للعين بس. زي ما قُلتلك، يا عزيزي، لأنه حرّك الكاميرا، عشان تخدم الحدوتة، بدل ما كانت مجرد أداة ثابتة وبتصور. بس «ويلز« مش بس حركها، دا حوّل الكاميرا بيزحف، بيطير، بيتسلل زي الشبح، الكاميرا بقت بتتحرك حركة مستحيلة، كأنها طائر،

[Source 2] Episode: هل Citizen Kane أفضل فيلم في التاريخ؟  الدحيح | Time: 722.178 - 747.294
كل المعاني بتحصل قُدّامك في لَقطة واحدة. طبعًا، التأثير اللي «ويلز« و«تولاند« بقى «أوبشن« عادي جدًا بس الفَرْق إن «ويلز« كان بيخترع دا عشان يقدر يعمل تركيز وعمق بصري، يخلّي عينك انت هي اللي تمسح الكادر وتختار هتركّز على إيه، كأن «ويلز« كان بيخلق قبل ما الـ«سوشيال ميديا« نفسها عشان كدا، الـ Deep Focus لم تعُد دي بقت فلسفة كاملة في الإخراج، العمق البصري بقى مرادف للعمق النفسي،

[Source 3] Episode: هل Citizen Kane أفضل فيلم في التاريخ؟  الدحيح | Time: 950.75 -

In [33]:
# ==========================================
# SYSTEM PROMPT
# ==========================================

SYSTEM_PROMPT = """
أنت مساعد ذكي يعتمد فقط على المعلومات الموجودة في السياق المسترجع.

قواعد مهمة:
- أجب باستخدام المعلومات الموجودة في السياق فقط.
- يمكنك إعادة صياغة وشرح المعلومات الموجودة بوضوح.
- لا تضف أي معلومات غير موجودة في السياق.
- إذا كان السياق لا يحتوي على معلومات كافية فعلًا، قل:
"لا أملك معلومات كافية للإجابة من البيانات المتاحة."
- يمكنك الإجابة بالعربية أو الإنجليزية حسب لغة السؤال.
- حاول أن تكون الإجابة واضحة ومختصرة.
"""

# ==========================================
# FINAL PROMPT CONSTRUCTION
# ==========================================

def build_prompt(query, context):
    """
    Builds the final prompt passed to the language model.
    """

    prompt = f"""
{SYSTEM_PROMPT}

السياق:
{context}

السؤال:
{query}

الإجابة:
"""

    return prompt

In [88]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_groq import ChatGroq
import os

# ==========================================
# INITIALIZE GEMINI MODEL
# ==========================================

llm = ChatGoogleGenerativeAI(
    model="gemini-2.0-flash",
    google_api_key=GOOGLE_API_KEY,
    temperature=0.3
)

print("Gemini model initialized successfully.")

# ==========================================
# INITIALIZE GROQ FALLBACK MODEL
# ==========================================

GROQ_API_KEY = os.getenv("GROQ_API_KEY")

groq_llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    groq_api_key=GROQ_API_KEY,
    temperature=0.3
)
print("Groq model initialized successfully.")

openrouter_llm = ChatOpenAI(

    model="meta-llama/llama-3.3-70b-instruct:free",

    api_key=OPENROUTER_API_KEY,

    base_url="https://openrouter.ai/api/v1",

    temperature=0.3

)

# ==========================================
# LLM CALL WITH RETRY
# ==========================================

def call_llm_with_retry(
    llm_model,
    prompt,
    max_retries=2
):
    """
    Calls an LLM with a retry mechanism.
    Retries API calls before failing.
    """

    last_error = None

    for attempt in range(max_retries + 1):

        try:
            response = llm_model.invoke(prompt)
            return response

        except Exception as e:
            last_error = e

    raise last_error

# ==========================================
# RAG ANSWER GENERATION PIPELINE
# ==========================================

def generate_answer(query, top_k=5):

    retrieved_chunks = retrieve_chunks(
        query,
        top_k=top_k
    )

    context = build_context(retrieved_chunks)

    # ==========================================
    # OUT-OF-DOMAIN CHECK
    # ==========================================

    if is_out_of_domain(retrieved_chunks):

        return {

            "query": query,

            "response": (
            "لا أملك معلومات كافية للإجابة من البيانات المتاحة.\n"
            "I do not have enough information to answer from the available data."
            ),

            "retrieved_chunks": retrieved_chunks,

            "context": "",

            "status": "out_of_domain",

            "model_used": None

        }

    prompt = build_prompt(query, context)

    # ==========================================
    # TRY GEMINI FIRST
    # ==========================================

    try:

        response = call_llm_with_retry(
            llm,
            prompt
        )

        return {
            "query": query,
            "response": response.content,
            "retrieved_chunks": retrieved_chunks,
            "context": context,
            "status": "success",
            "model_used": "gemini-2.0-flash"
        }

    # ==========================================
    # FALLBACK TO GROQ
    # ==========================================

    except Exception:

        try:

            response = call_llm_with_retry(
                groq_llm,
                prompt
            )

            return {
                "query": query,
                "response": response.content,
                "retrieved_chunks": retrieved_chunks,
                "context": context,
                "status": "success",
                "model_used": "llama-3.3-70b-versatile"
            }

        except Exception as e:

            return {
                "query": query,
                "response": (
                    "حدث خطأ أثناء استدعاء نماذج اللغة. "
                    "يرجى المحاولة لاحقًا."
                ),
                "retrieved_chunks": [],
                "context": "",
                "status": "error",
                "error_message": str(e)
            }

Gemini model initialized successfully.
Groq model initialized successfully.


In [44]:
# ==========================================
# LANGCHAIN RAG CHAIN WRAPPER
# ==========================================

from langchain_core.runnables import RunnableLambda

def langchain_rag_function(query):
    """
    LangChain-compatible wrapper around the custom RAG pipeline.
    This keeps our FAISS hybrid retriever while exposing the RAG flow
    as a LangChain Runnable chain.
    """

    return generate_answer(
        query=query,
        top_k=3
    )


rag_chain = RunnableLambda(langchain_rag_function)

print("LangChain RAG chain initialized successfully.")


LangChain RAG chain initialized successfully.


In [45]:
result = generate_answer(
    "What is Deep Focus?",
    top_k=3
)

print("Status:", result["status"])
print("Model:", result.get("model_used"))
print(result["response"])

Status: success
Model: llama-3.3-70b-versatile
Deep Focus هي تقنية في التصوير السينمائي تتيح للكاميرا التقاط الصورة بعمق بصري كبير، بحيث تكون جميع الأشياء في الصورة واضحة ومحددة، من الأمام إلى الخلف. وتُستخدم هذه التقنية لخلق تأثير عميق نفسي، حيث تصبح الصورة مرآة للعقل، لا للعين فقط.


In [39]:
# ==========================================
# CONTEXT WINDOW STRATEGIES
# ==========================================

def get_context_window(history, strategy="sliding_window", max_turns=3):
    """
    Returns conversation history using different context window strategies.

    Strategies:
    - full_history: use all previous turns
    - sliding_window: use last N turns
    - strict_truncation: keep first turn + last N-1 turns
    - summarized_history: use a simple summary placeholder + recent turns
    """

    if strategy == "full_history":
        selected_history = history

    elif strategy == "sliding_window":
        selected_history = history[-max_turns:]

    elif strategy == "strict_truncation":
        if len(history) <= max_turns:
            selected_history = history
        else:
            selected_history = [history[0]] + history[-(max_turns - 1):]

    elif strategy == "summarized_history":
        if len(history) <= max_turns:
            selected_history = history
        else:
            summary_turn = {
                "user": "Conversation summary",
                "assistant": "Previous conversation discussed earlier user questions and assistant answers."
            }
            selected_history = [summary_turn] + history[-(max_turns - 1):]

    else:
        selected_history = history[-max_turns:]

    return selected_history


# ==========================================
# MULTI-TURN CHAT MEMORY
# ==========================================

conversation_history = []

def format_chat_history(
    history,
    max_turns=3,
    strategy="sliding_window"
):
    """
    Formats chat history according to the selected context window strategy.
    """

    selected_history = get_context_window(
        history,
        strategy=strategy,
        max_turns=max_turns
    )

    formatted_history = []

    for turn in selected_history:
        formatted_history.append(
            f"User: {turn['user']}\nAssistant: {turn['assistant']}"
        )

    return "\n\n".join(formatted_history)


def build_chat_prompt(query, context, chat_history):
    """
    Builds a multi-turn prompt using retrieved context
    plus recent conversation history.
    """

    prompt = f"""
{SYSTEM_PROMPT}

سجل المحادثة السابق:
{chat_history}

السياق المسترجع:
{context}

السؤال الحالي:
{query}

الإجابة:
"""

    return prompt


def chat_with_memory(query, top_k=5, max_turns=3, memory_strategy="sliding_window"):
    """
    Multi-turn RAG chatbot with sliding-window memory.
    """

    # ==========================================
    # HISTORY-AWARE RETRIEVAL QUERY
    # ==========================================

    retrieval_query = query

    if conversation_history:
        last_user_query = conversation_history[-1]["user"]

        retrieval_query = (
            last_user_query + " " + query
        )

    retrieved_chunks = retrieve_chunks(
        retrieval_query,
        top_k=top_k
    )

    # ==========================================
    # OUT-OF-DOMAIN CHECK
    # ==========================================

    if is_out_of_domain(retrieved_chunks):

        return {

            "query": query,

            "response": (
            "لا أملك معلومات كافية للإجابة من البيانات المتاحة.\n"
            "I do not have enough information to answer from the available data."
            ),

            "retrieved_chunks": retrieved_chunks,

            "context": "",

            "chat_history": "",

            "status": "out_of_domain",

            "model_used": None

        }
    context = build_context(retrieved_chunks)

    chat_history = format_chat_history(
    conversation_history,
    max_turns=max_turns,
    strategy=memory_strategy
    )

    prompt = build_chat_prompt(
        query,
        context,
        chat_history
    )

    try:
        response = call_llm_with_retry(
            llm,
            prompt
        )
        model_used = "gemini-2.0-flash"

    except Exception:
        response = call_llm_with_retry(
            groq_llm,
            prompt
        )
        model_used = "llama-3.3-70b-versatile"

    answer = response.content

    conversation_history.append({
        "user": query,
        "assistant": answer
    })

    return {
        "query": query,
        "response": answer,
        "retrieved_chunks": retrieved_chunks,
        "context": context,
        "chat_history": chat_history,
        "model_used": model_used
    }

In [41]:
conversation_history = []

result1 = chat_with_memory(
    "ما هي طائرة F-35؟",
    top_k=5,
    memory_strategy="full_history"
)

print(result1["response"])

result2 = chat_with_memory(
    "لماذا يصعب تتبعها؟",
    top_k=5,
    memory_strategy="sliding_window"
)

print(result2["response"])

الطائرة F-35 هي طائرة حربية متقدمة تتميز بتقنيات متطورة مثل رادار APG-81 وبرمجيات عبقرية تجمع وتحلل البيانات، وتتميز أيضًا بخوذة تقدم معلومات دقيقة للطيار. وهي تُعتبر جزءًا من نظام حربي متكامل يسمح للطيار بالاستفادة من المعلومات المجمعة لتحقيق الأهداف العسكرية.
تتميز طائرة F-35 بنظام اتصال يصعب التقاطه ويصعب كذلك تتبعه.


In [51]:
# ==========================================
# SELECT QA FILES FOR CURRENT RAG DATASET
# ==========================================

SELECTED_QA_FILES = [
    "f35_qa_dataset.json",
    "samurai_qa_dataset.json",
    "octopus_qa_dataset.json",
    "citizen_kane_qa_dataset.json"
]

qa_files = [
    QA_DIR / file_name
    for file_name in SELECTED_QA_FILES
]

print("Selected QA files:")
for file in qa_files:
    print(file.name, "exists:", file.exists())

Selected QA files:
f35_qa_dataset.json exists: True
samurai_qa_dataset.json exists: True
octopus_qa_dataset.json exists: True
citizen_kane_qa_dataset.json exists: True


In [52]:
# ==========================================
# LOAD QA EVALUATION SET FROM MS2 FILES
# ==========================================

def load_qa_evaluation_set(qa_files, max_questions_per_file=5):
    evaluation_set = []

    for qa_file in qa_files:

        with open(qa_file, "r", encoding="utf-8") as f:
            qa_data = json.load(f)

        count = 0

        for article in qa_data.get("data", []):
            for paragraph in article.get("paragraphs", []):
                for qa in paragraph.get("qas", []):

                    if count >= max_questions_per_file:
                        break

                    answers = qa.get("answers", [])

                    if not answers:
                        continue

                    evaluation_set.append({
                        "source_file": qa_file.name,
                        "question": qa["question"],
                        "expected_answer": answers[0]["text"]
                    })

                    count += 1

                if count >= max_questions_per_file:
                    break

            if count >= max_questions_per_file:
                break

    return evaluation_set


evaluation_set = load_qa_evaluation_set(
    qa_files,
    max_questions_per_file=5
)

print("Total evaluation questions:", len(evaluation_set))

display(pd.DataFrame(evaluation_set).head())

Total evaluation questions: 20


,source_file,question,expected_answer
0,f35_qa_dataset.json,ما اسم الطائرة التي تدور حولها الحلقة؟,F-35
1,f35_qa_dataset.json,في أي شهر وسنة جرت تجربة الطائرة المذكورة؟,يوليو 2011
2,f35_qa_dataset.json,أين كان يقف المقدم إيريك سميث أثناء التجربة؟,في Hangar واسع في ولاية تكساس
3,f35_qa_dataset.json,إلى أين كانت رحلة الاختبار للطائرة؟,من المصنع لقاعدة موجودة في فلوريدا
4,f35_qa_dataset.json,لماذا كانت تجربة الطائرة حساسة إعلاميًا؟,ممكن تأثر على مبيعات لوكهيد مارتن، ودي مبيعات ...


In [76]:
# ==========================================
# EVALUATION METRICS
# ==========================================

def semantic_similarity(text1, text2):
    """
    Semantic correctness:
    compares generated answer with expected answer using embeddings.
    """

    embeddings_eval = embedding_model.encode(
        [text1, text2],
        convert_to_numpy=True,
        normalize_embeddings=True
    )

    return float(np.dot(embeddings_eval[0], embeddings_eval[1]))


def grounding_score(answer, context):
    """
    Grounding:
    measures how many answer terms appear in the retrieved context.
    """

    answer_terms = extract_query_terms(answer)
    context_text = context.lower()

    if not answer_terms:
        return 0.0

    matches = sum(
        1 for term in answer_terms
        if term in context_text
    )

    return matches / len(answer_terms)


def quality_score(answer):
    """
    Generation quality:
    simple heuristic for non-empty, non-error, non-rejection answers.
    """

    if not answer or len(answer.strip()) < 20:
        return 0.0

    bad_phrases = [
        "حدث خطأ",
        "لا أملك معلومات كافية",
        "I do not have enough information"
    ]

    if any(phrase in answer for phrase in bad_phrases):
        return 0.0

    return 1.0

In [89]:
# ==========================================
# GENERATE ANSWER WITH SELECTED MODEL
# ==========================================

def generate_answer_with_model(query, model_name="gemini", top_k=3):
    retrieved_chunks = retrieve_chunks(
        query,
        top_k=top_k
    )

    context = build_context(retrieved_chunks)

    if is_out_of_domain(retrieved_chunks):
        return {
            "query": query,
            "response": (
                "لا أملك معلومات كافية للإجابة من البيانات المتاحة.\n"
                "I do not have enough information to answer from the available data."
            ),
            "retrieved_chunks": retrieved_chunks,
            "context": context,
            "status": "out_of_domain",
            "model_used": model_name,
            "error_message": ""
        }

    prompt = build_prompt(query, context)

    try:
        if model_name == "gemini":
            response = call_llm_with_retry(
                llm,
                prompt
            )
            used_model = "gemini-2.0-flash"

        elif model_name == "groq":
            response = call_llm_with_retry(
                groq_llm,
                prompt
            )
            used_model = "llama-3.3-70b-versatile"


        elif model_name == "openrouter":
            response = call_llm_with_retry(
                openrouter_llm,
                prompt
            )
            used_model = "openrouter-llama-3.3-70b"


        else:
            raise ValueError("model_name must be either 'gemini' or 'groq'")

        return {
            "query": query,
            "response": response.content,
            "retrieved_chunks": retrieved_chunks,
            "context": context,
            "status": "success",
            "model_used": used_model,
            "error_message": ""
        }

    except Exception as e:
        return {
            "query": query,
            "response": "",
            "retrieved_chunks": retrieved_chunks,
            "context": context,
            "status": "llm_error",
            "model_used": model_name,
            "error_message": str(e)
        }

In [ ]:
# ==========================================
# RUN EVALUATION ON 3 LLMS (10 QUESTIONS)
# ==========================================

evaluation_logs = []

evaluation_set_small = evaluation_set[:10]

for item in evaluation_set_small:

    for model_name in ["gemini", "groq", "openrouter"]:

        result = generate_answer_with_model(
            item["question"],
            model_name=model_name,
            top_k=5
        )

        answer = result["response"]
        context = result["context"]

        evaluation_logs.append({
            "source_file": item["source_file"],
            "question": item["question"],
            "expected_answer": item["expected_answer"],
            "model": result.get("model_used"),
            "answer": answer,
            "semantic_correctness": semantic_similarity(
                answer,
                item["expected_answer"]
            ) if answer else 0.0,
            "grounding": grounding_score(
                answer,
                context
            ) if answer else 0.0,
            "quality": quality_score(answer),
            "status": result["status"],
            "error_message": result.get("error_message", "")
        })

evaluation_df = pd.DataFrame(evaluation_logs)

display(evaluation_df)

KeyboardInterrupt: 

In [95]:
# ==========================================
# CREATE STREAMLIT INTERFACE FILE
# ==========================================

streamlit_code = r'''
import streamlit as st
import pandas as pd

# ==========================================
# PAGE CONFIG
# ==========================================

st.set_page_config(
    page_title="Arabic RAG Chatbot - MS3",
    page_icon="💬",
    layout="wide"
)

st.title("💬 Arabic RAG Chatbot")
st.caption("Milestone 3: Retrieval-Augmented Generation over Arabic transcripts")

# ==========================================
# SESSION STATE
# ==========================================

if "messages" not in st.session_state:
    st.session_state.messages = []

if "evaluation_logs" not in st.session_state:
    st.session_state.evaluation_logs = []

# ==========================================
# SIDEBAR SETTINGS
# ==========================================

st.sidebar.header("Settings")

top_k = st.sidebar.slider(
    "Top-K Retrieved Chunks",
    min_value=1,
    max_value=8,
    value=5
)

memory_strategy = st.sidebar.selectbox(
    "Memory Strategy",
    [
        "sliding_window",
        "full_history",
        "strict_truncation",
        "summarized_history"
    ]
)

max_turns = st.sidebar.slider(
    "Max Memory Turns",
    min_value=1,
    max_value=10,
    value=3
)

show_logs = st.sidebar.checkbox(
    "Show Retrieval Logs",
    value=True
)

# ==========================================
# DISPLAY CHAT HISTORY
# ==========================================

for message in st.session_state.messages:
    with st.chat_message(message["role"]):
        st.markdown(message["content"])

# ==========================================
# CHAT INPUT
# ==========================================

user_query = st.chat_input("Ask a question about the selected Arabic transcripts...")

if user_query:

    st.session_state.messages.append({
        "role": "user",
        "content": user_query
    })

    with st.chat_message("user"):
        st.markdown(user_query)

    with st.chat_message("assistant"):

        with st.spinner("Retrieving context and generating answer..."):

            result = chat_with_memory(
                user_query,
                top_k=top_k,
                max_turns=max_turns,
                memory_strategy=memory_strategy
            )

            answer = result["response"]

            st.markdown(answer)

            st.session_state.messages.append({
                "role": "assistant",
                "content": answer
            })

            st.session_state.evaluation_logs.append({
                "query": result["query"],
                "response": result["response"],
                "model_used": result.get("model_used"),
                "status": result.get("status", "success"),
                "memory_strategy": memory_strategy,
                "top_k": top_k
            })

            if show_logs:

                with st.expander("Retrieved Context / Logs"):

                    st.write("Model used:", result.get("model_used"))
                    st.write("Status:", result.get("status", "success"))
                    st.write("Memory strategy:", memory_strategy)

                    retrieved_chunks = result.get("retrieved_chunks", [])

                    for i, chunk in enumerate(retrieved_chunks, start=1):
                        st.markdown(f"### Source {i}")
                        st.write("Episode:", chunk["episode"])
                        st.write(
                            "Time:",
                            chunk["start_timestamp"],
                            "→",
                            chunk["end_timestamp"]
                        )
                        st.write("Final score:", round(chunk["final_score"], 4))
                        st.write(chunk["chunk_text"])

# ==========================================
# LOGS PANEL
# ==========================================

st.divider()

st.subheader("Session Logs")

if st.session_state.evaluation_logs:
    logs_df = pd.DataFrame(st.session_state.evaluation_logs)
    st.dataframe(logs_df, use_container_width=True)

    csv = logs_df.to_csv(index=False).encode("utf-8-sig")

    st.download_button(
        label="Download session logs as CSV",
        data=csv,
        file_name="streamlit_session_logs.csv",
        mime="text/csv"
    )
else:
    st.info("No logs yet. Start chatting to generate logs.")
'''

with open("streamlit_app.py", "w", encoding="utf-8") as f:
    f.write(streamlit_code)

print("streamlit_app.py created successfully.")

streamlit_app.py created successfully.
